# LeRobot to PhysicalAI: Portable Policy Inference

This notebook demonstrates the **cross-framework** workflow: training or preparing a policy in LeRobot, exporting it to the universal `policy_package` format, and deploying it with the `PhysicalAI InferenceModel`. 

Both frameworks share the same converged manifest format, enabling seamless interop between research-oriented training (LeRobot) and deployment-focused inference (PhysicalAI).

### Workflow
1. **Train/Prepare (LeRobot)**: Define and initialize a policy.
2. **Export (LeRobot)**: Export the policy to a standardized package with a `manifest.json`.
3. **Deploy (PhysicalAI)**: Load the package using `InferenceModel` for production-ready inference.

### Prerequisites
- `lerobot`
- `physicalai[inference]`
- Optional: `openvino`

## 1. Export a Trained Policy

We start by setting up a lightweight ACT (Action Chunking Transformer) policy in LeRobot and exporting it to ONNX. We use simulated weights for this demonstration.

In [ ]:
import torch
from lerobot.configs.types import FeatureType, NormalizationMode, PolicyFeature
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.utils.constants import ACTION, OBS_STATE
from lerobot.export import export_policy

input_features = {
    OBS_STATE: PolicyFeature(type=FeatureType.STATE, shape=(6,)),
    "observation.images.top": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 84, 84)),
}
output_features = {
    ACTION: PolicyFeature(type=FeatureType.ACTION, shape=(6,)),
}

config = ACTConfig(
    device="cpu",
    chunk_size=10,
    n_action_steps=10,
    dim_model=64,
    n_heads=2,
    dim_feedforward=128,
    n_encoder_layers=2,
    n_decoder_layers=2,
    n_vae_encoder_layers=2,
    use_vae=False,
    latent_dim=16,
    vision_backbone="resnet18",
    pretrained_backbone_weights=None,
    input_features=input_features,
    output_features=output_features,
    normalization_mapping={
        "VISUAL": NormalizationMode.MEAN_STD,
        "STATE": NormalizationMode.MEAN_STD,
        "ACTION": NormalizationMode.MEAN_STD,
    },
)

policy = ACTPolicy(config)
policy.eval()

batch = {
    "observation.state": torch.randn(1, 6),
    "observation.images.top": torch.randn(1, 3, 84, 84),
}

package_path = export_policy(
    policy,
    "./exported_act_for_physicalai",
    backend="onnx",
    example_batch=batch,
    include_normalization=False,
)
print(f"Exported to: {package_path}")

## 2. Inspect the Manifest

The exported package contains a `manifest.json` that follows the converged specification. This manifest describes the policy architecture, the runtime runner, and the model artifacts.

In [ ]:
import json
from pathlib import Path

manifest_path = Path(package_path) / "manifest.json"
with open(manifest_path, "r") as f:
    manifest = json.load(f)

print(f"Format: {manifest['format']}")
print(f"Version: {manifest['version']}")
print(f"Policy: {manifest['policy']['name']}")
print(f"Runner: {manifest['model']['runner']['type']}")
print(f"Artifacts: {list(manifest['model']['artifacts'].keys())}")

## 3. Load with PhysicalAI InferenceModel

PhysicalAI provides a high-level `InferenceModel` class that handles manifest parsing and runner initialization automatically.

In [ ]:
from physicalai.inference import InferenceModel

model = InferenceModel(package_path, backend="onnx", device="cpu")
print(model)
print(f"Manifest format: {model.manifest.format}")
print(f"Policy: {model.manifest.policy.name}")
print(f"Runner: {model.manifest.model.runner.type}")

## 4. Run Inference

The `InferenceModel` can be called directly with a dictionary of observations. For chunking policies like ACT, it returns the full predicted chunk.

In [ ]:
import numpy as np

observation = {
    "observation.state": np.random.randn(1, 6).astype(np.float32),
    "observation.images.top": np.random.randn(1, 3, 84, 84).astype(np.float32),
}

# Full model call returns dict
outputs = model(observation)
print(f"Output keys: {list(outputs.keys())}")
print(f"Action shape: {outputs['action'].shape}")

## 5. select_action() for Control Loops

For real-time control, `select_action()` is a convenience method that manages the internal action queue. It extracts and returns a single action from the current chunk or triggers a new model pass if the queue is empty.

In [ ]:
# select_action() extracts and returns a single action from the chunk queue
action = model.select_action(observation)
print(f"Action shape: {action.shape}")
print(f"Action dtype: {action.dtype}")

### Simulated Control Loop

The `reset()` method clears any remaining actions in the queue, ensuring the next `select_action()` call triggers a fresh inference pass.

In [ ]:
model.reset()  # Reset action queue

for step in range(15):
    action = model.select_action(observation)
    print(f"Step {step:2d}: action={action[:3]}")
    # In real deployment:
    # env.step(action)
    # observation = get_new_observation(env)

## 6. Context Manager

`InferenceModel` supports the context manager pattern for clean resource handling.

In [ ]:
with InferenceModel(package_path, backend="onnx", device="cpu") as model:
    action = model.select_action(observation)
    print(f"Action: {action.shape}")

## 7. OpenVINO Backend

Since the package format is backend-agnostic, you can switch from ONNX to OpenVINO simply by changing the `backend` parameter.

In [ ]:
try:
    ov_model = InferenceModel(package_path, backend="openvino", device="cpu")
    ov_output = ov_model(observation)
    
    onnx_model = InferenceModel(package_path, backend="onnx", device="cpu")
    onnx_output = onnx_model(observation)
    
    np.testing.assert_allclose(
        onnx_output["action"], ov_output["action"], atol=1e-5
    )
    print("ONNX and OpenVINO outputs match! ✅")
except ImportError:
    print("OpenVINO not installed - skipping")

## 8. Cross-Framework Parity

Finally, we verify that the same package produces identical results whether loaded via LeRobot's internal runner or PhysicalAI's `InferenceModel`.

In [ ]:
from lerobot.export import load_exported_policy

lerobot_runner = load_exported_policy(package_path, backend="onnx", device="cpu")
lerobot_chunk = lerobot_runner.predict_action_chunk(observation)

physicalai_model = InferenceModel(package_path, backend="onnx", device="cpu")
physicalai_output = physicalai_model(observation)

np.testing.assert_allclose(
    lerobot_chunk, physicalai_output["action"], atol=1e-6
)
print("LeRobot and PhysicalAI produce identical results! ✅")

## 9. Supported Policies

The `InferenceModel` supports all policies that can be exported to the `policy_package` format:

| Policy | Type | Description |
| :--- | :--- | :--- |
| **ACT** | Chunking Transformer | Robust action sequences from visual/state inputs. |
| **Diffusion** | Generative | High-precision paths via denoising. |
| **TDMPC2** | Model-Predictive | Efficient latent planning and control. |
| **VQ-BeT** | Vector-Quantized | Discrete action clusters for complex behaviors. |
| **Pi0** | Flow-Matching | State-of-the-art vision-language-action foundation model. |

## 10. Cleanup

In [ ]:
import shutil
if Path(package_path).exists():
    shutil.rmtree(package_path)
    print("Cleaned up exported package.")